# Main text simulation study

This notebook is intended for the main simulation section.  It uses three DGPs and keeps the experiments grouped into three manuscript subsections.  Every subsection estimates both ATE and ATT whenever the wrapper succeeds.

In [ ]:
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings("once")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Make the experiment helper importable whether Jupyter is launched from the
# repository root or from notebooks/experiments.
for _candidate in [Path.cwd(), Path.cwd() / "notebooks" / "experiments"]:
    if (_candidate / "grr_experiment_utils.py").exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

# Local experiment helpers. These live beside the notebooks and do not modify src/genriesz.
from grr_experiment_utils import *
from genriesz import (
    grr_ate, grr_att, ATEFunctional, ATTFunctional,
    BregmanGenerator, SquaredGenerator, UKLGenerator, BKLGenerator, BPGenerator,
    PolynomialBasis, TreatmentInteractionBasis,
)

# Optional random-forest leaf basis used in model-comparison experiments.
try:
    from sklearn.ensemble import RandomForestRegressor
    from genriesz.sklearn_basis import RandomForestLeafBasis
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    print("scikit-learn is not available; random-forest basis cells will fall back to RKHS.", exc)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

In [ ]:
FAST_MODE = True
POLYNOMIAL_DEGREE = 1 if FAST_MODE else 2
DOWNLOAD_DATA = False if FAST_MODE else True
os.environ["GRR_ALLOW_REMOTE_DATA"] = "1" if DOWNLOAD_DATA else "0"

# Every analysis estimates both targets when the wrapper supports them.
# If one target fails for a particular method, the failure row is kept and the other target is displayed.
ESTIMANDS = ("ate", "att")

N_REPS = 1 if FAST_MODE else 200
N = 40 if FAST_MODE else 3000
FOLDS = 2 if FAST_MODE else 5
MAX_ITER = 25 if FAST_MODE else 500
N_FEATURES = 4 if FAST_MODE else 120

LOSS_GRID = [("SQ", None), ("UKL", None), ("BKL", None), ("BP", 0.5)]
LAMBDA_MAIN = 1e-2
ESTIMATORS_ALL = ("ra", "rw", "arw", "tmle")
print(display_mode_banner(FAST_MODE))
DGP_NAMES = [
    "DGP1 nonlinear heterogeneous",
    "DGP2 weak overlap",
    "DGP3 Kang-Schafer misspecification",
]
MODEL_GRID = ["rkhs", "polynomial", "rf"]

TABLE_TITLE_SECTION_1 = "Table S-main-1. Compatible loss--link pairs by estimand, DGP, and model."
FIGURE_TITLE_SECTION_1 = "Boxplots of ARW squared error: compatible loss--link pairs"
TABLE_TITLE_SECTION_2 = "Table S-main-2. Incompatible loss--link pairs with RKHS basis."
FIGURE_TITLE_SECTION_2 = "Boxplots of squared error: incompatible loss--link pairs"
FIGURE_TITLE_SECTION_3 = "Regularization path: MSE versus Riesz penalty"

In [ ]:
def dgp_factory(name, *, n, seed):
    """Three DGPs used throughout the main simulation study."""
    if name == "DGP1 nonlinear heterogeneous":
        return make_ate_data(n=n, d=8, kappa=1.0, heterogeneous=True, seed=seed, noise_sd=1.0)
    if name == "DGP2 weak overlap":
        return make_ate_data(n=n, d=8, kappa=2.5, heterogeneous=True, seed=seed, noise_sd=1.0)
    if name == "DGP3 Kang-Schafer misspecification":
        return make_kang_schafer_data(n=n, seed=seed, tau=1.0)
    raise ValueError(name)


def basis_for_model(model, *, seed=0, n_features=N_FEATURES, sigma=1.0):
    """Editable basis map used by the GRR experiments."""
    model = str(model).lower()
    if model in {"rkhs", "gaussian"}:
        return make_treatment_basis("rkhs", n_features=n_features, sigma=sigma, seed=seed)
    if model in {"poly", "polynomial"}:
        return make_treatment_basis("poly", degree=POLYNOMIAL_DEGREE, n_features=n_features, sigma=sigma, seed=seed)
    if model in {"rff", "fourier", "random_fourier"}:
        return make_treatment_basis("rff", n_features=n_features, sigma=sigma, seed=seed)
    if model in {"rf", "random_forest"}:
        if SKLEARN_AVAILABLE:
            rf = RandomForestRegressor(n_estimators=8 if FAST_MODE else 80, max_depth=3 if FAST_MODE else 5,
                                       min_samples_leaf=5, random_state=seed)
            return TreatmentInteractionBasis(base_basis=RandomForestLeafBasis(rf, include_bias=True, normalize=True))
        return make_treatment_basis("rkhs", n_features=n_features, sigma=sigma, seed=seed)
    raise ValueError(model)


def loss_label(loss, omega=None):
    return loss if omega is None else f"{loss}({omega:g})"


def add_metadata(df, **kwargs):
    out = df.copy()
    for k, v in kwargs.items():
        out[k] = v
    return out


def clean_results(df):
    if "status" in df.columns:
        status = df["status"].fillna("ok")
    else:
        status = pd.Series("ok", index=df.index)
    return df[status.eq("ok") & df["estimator"].ne("failed")].copy()


def mc_table(df, group_cols, estimator_filter=None):
    d = clean_results(df)
    if estimator_filter is not None:
        d = d[d["estimator"].isin(list(estimator_filter))]
    return summarize_mc(d, group_cols)


def boxplot_metric(df, *, group_col, metric, title, estimator="arw", rotate=45, ylim=None):
    d = clean_results(df)
    if estimator is not None:
        d = d[d["estimator"] == estimator]
    labels = list(d[group_col].dropna().astype(str).unique())
    data = [d.loc[d[group_col].astype(str) == lab, metric].dropna().to_numpy() for lab in labels]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(labels)), 4.5))
    ax.boxplot(data, labels=labels, showmeans=True)
    ax.set_title(title)
    ax.set_xlabel(group_col)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=rotate)
    if ylim is not None:
        ax.set_ylim(*ylim)
    fig.tight_layout()
    plt.show()
    return fig, ax


def run_one(data, *, estimand, loss, omega, basis, lam, cross_fit, folds, estimators=ESTIMATORS_ALL,
            penalty="l2", max_iter=MAX_ITER):
    """Fit ATE or ATT. Failures are returned as rows so tables remain complete."""
    try:
        df = fit_grr_estimand(
            data, estimand=estimand, loss=loss, omega=omega, basis=basis, lam=lam,
            cross_fit=cross_fit, folds=folds, estimators=estimators, penalty=penalty,
            max_iter=max_iter,
        )
        df["status"] = "ok"
        return df
    except Exception as exc:
        theta = true_theta_for_estimand(data, estimand)
        row = {
            "estimand": estimand.upper(), "estimator": "failed", "status": type(exc).__name__,
            "message": str(exc)[:240], "true_theta": theta,
        }
        return pd.DataFrame([row])

## 1. Compatible regressor-balancing loss--link pairs

The generator-induced link is used for every loss.  The balancing dictionary is the treatment-interaction basis for both ATE and ATT.  Models are RKHS, polynomial basis, and random-forest leaf basis.

In [ ]:
rows = []
for dgp_name in (DGP_NAMES[:1] if FAST_MODE else DGP_NAMES):
    for rep in range(N_REPS):
        data = dgp_factory(dgp_name, n=N, seed=1000 + 97 * rep)
        for estimand in ESTIMANDS:
            for model_name in MODEL_GRID:
                for loss, omega in LOSS_GRID:
                    basis = basis_for_model(model_name, seed=rep, n_features=N_FEATURES, sigma=1.0)
                    df = run_one(
                        data, estimand=estimand, loss=loss, omega=omega, basis=basis, lam=LAMBDA_MAIN,
                        cross_fit=True, folds=FOLDS, estimators=ESTIMATORS_ALL, max_iter=MAX_ITER,
                    )
                    df = add_metadata(df, dgp=dgp_name, rep=rep, model=model_name,
                                      loss=loss_label(loss, omega), lambda_value=LAMBDA_MAIN,
                                      cross_fit=True, experiment="main-compatible")
                    rows.append(df)

main_compatible = pd.concat(rows, ignore_index=True)
print(TABLE_TITLE_SECTION_1)
main_compatible_table = mc_table(main_compatible, ["estimand", "dgp", "model", "loss", "estimator"])
display(safe_display_frame(main_compatible_table, n=120))

In [ ]:
plot_df = clean_results(main_compatible).copy()
plot_df = plot_df[plot_df["estimator"] == "arw"]
plot_df["method"] = plot_df["estimand"] + " | " + plot_df["dgp"] + " | " + plot_df["model"] + " | " + plot_df["loss"]
boxplot_metric(plot_df, group_col="method", metric="squared_error", title=FIGURE_TITLE_SECTION_1, estimator=None, rotate=85)

## 2. Incompatible loss--link pairs

This subsection intentionally breaks dual linearity.  The examples are BKL loss with a logit-type inverse link, SQ loss with a logit-type inverse link, and UKL loss with a linear inverse link.  These are diagnostic experiments, not recommended GRR specifications.  The basis is RKHS and cross-fitting is enabled.

In [ ]:
def make_incompatible_generator(pair):
    """Construct intentionally mismatched loss--link pairs for Section 2."""
    pair = pair.lower()
    if pair == "bkl_logit":
        base = BKLGenerator(C=1.0, branch_fn=branch_treated)
        base_name = "BKL loss + logit link"
        link = "logit"
    elif pair == "sq_logit":
        base = SquaredGenerator(C=0.0)
        base_name = "SQ loss + logit link"
        link = "logit"
    elif pair == "ukl_linear":
        base = UKLGenerator(C=1.0, branch_fn=branch_treated)
        base_name = "UKL loss + linear link"
        link = "linear"
    else:
        raise ValueError(pair)

    def inv_logit(X, v):
        X = np.asarray(X, float)
        z = np.clip(np.asarray(v, float), -4.0, 4.0)
        return np.where(X[:, 0] >= 0.5, 1.0 + np.exp(-z), -1.0 - np.exp(z))

    def inv_linear(X, v):
        X = np.asarray(X, float)
        s = np.where(X[:, 0] >= 0.5, 1.0, -1.0)
        t = 1.5 + 0.1 * s * np.clip(np.asarray(v, float), -4.0, 4.0)
        t = np.maximum(t, 1.0001)
        return s * t

    return BregmanGenerator(
        g=base.g,
        grad=base.grad,
        grad2=base.grad2,
        inv_grad=inv_logit if link == "logit" else inv_linear,
        name=base_name,
    )

INCOMPATIBLE_PAIRS = [
    ("BKL+logit MLE", "bkl_logit"),
    ("SQ+logit", "sq_logit"),
    ("UKL+linear", "ukl_linear"),
]

rows = []
for dgp_name in (DGP_NAMES[:1] if FAST_MODE else DGP_NAMES):
    for rep in range(N_REPS):
        data = dgp_factory(dgp_name, n=N, seed=2000 + 97 * rep)
        for estimand in ESTIMANDS:
            for pair_label, pair_key in INCOMPATIBLE_PAIRS:
                try:
                    basis = basis_for_model("rkhs", seed=rep, n_features=N_FEATURES, sigma=1.0)
                    gen = make_incompatible_generator(pair_key)
                    wrapper = grr_ate if estimand == "ate" else grr_att
                    res = wrapper(
                        X=data["X"], Y=data["Y"], basis=basis, generator=gen,
                        cross_fit=True, folds=FOLDS, riesz_lam=LAMBDA_MAIN,
                        estimators=ESTIMATORS_ALL, max_iter=MAX_ITER,
                    )
                    df = result_to_frame(res, true_theta=true_theta_for_estimand(data, estimand))
                    df["estimand"] = estimand.upper(); df["status"] = "ok"
                except Exception as exc:
                    df = pd.DataFrame([{"estimand": estimand.upper(), "estimator": "failed", "status": type(exc).__name__, "message": str(exc)[:240]}])
                df = add_metadata(df, dgp=dgp_name, rep=rep, pair=pair_label, loss=pair_label,
                                  lambda_value=LAMBDA_MAIN, cross_fit=True, experiment="main-incompatible")
                rows.append(df)

main_incompatible = pd.concat(rows, ignore_index=True)
print(TABLE_TITLE_SECTION_2)
main_incompatible_table = mc_table(main_incompatible, ["estimand", "dgp", "pair", "estimator"])
display(safe_display_frame(main_incompatible_table, n=120))

In [ ]:
plot_df = clean_results(main_incompatible).copy()
plot_df = plot_df[plot_df["estimator"].isin(["rw", "arw", "tmle"])]
plot_df["method"] = plot_df["estimand"] + " | " + plot_df["pair"] + " | " + plot_df["estimator"]
boxplot_metric(plot_df, group_col="method", metric="squared_error", title=FIGURE_TITLE_SECTION_2, estimator=None, rotate=80)

## 3. Regularization path with RKHS basis

This subsection uses compatible loss--link pairs, an RKHS basis, and varies the Riesz regularization parameter.  Both cross-fitted and non-cross-fitted runs are computed for ATE and ATT.  The figure uses MSE on the y-axis and the regularization parameter on the x-axis.

In [ ]:
LAMBDA_GRID_PATH = [1e-3, 1e-2, 1e-1] if FAST_MODE else [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
PATH_DGP_NAMES = DGP_NAMES[:1] if FAST_MODE else DGP_NAMES
rows = []
for dgp_name in PATH_DGP_NAMES:
    for rep in range(N_REPS):
        data = dgp_factory(dgp_name, n=N, seed=3000 + 97 * rep)
        for estimand in ESTIMANDS:
            for cross_fit in [False, True]:
                for lam in LAMBDA_GRID_PATH:
                    for loss, omega in LOSS_GRID:
                        basis = basis_for_model("rkhs", seed=rep, n_features=N_FEATURES, sigma=1.0)
                        df = run_one(
                            data, estimand=estimand, loss=loss, omega=omega, basis=basis, lam=lam,
                            cross_fit=cross_fit, folds=FOLDS, estimators=("rw", "arw"), max_iter=MAX_ITER,
                        )
                        df = add_metadata(df, dgp=dgp_name, rep=rep, loss=loss_label(loss, omega),
                                          lambda_value=lam, cross_fit=cross_fit, model="rkhs",
                                          experiment="main-regularization-path")
                        rows.append(df)

lambda_path_results = pd.concat(rows, ignore_index=True)
lambda_path_table = mc_table(lambda_path_results, ["estimand", "dgp", "cross_fit", "lambda_value", "loss", "estimator"])
display(safe_display_frame(lambda_path_table, n=120))

In [ ]:
path_plot = lambda_path_table[lambda_path_table["estimator"] == "arw"].copy()
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, (estimand, cross_fit) in zip(axes.ravel(), [("ATE", False), ("ATE", True), ("ATT", False), ("ATT", True)]):
    dd = path_plot[(path_plot["estimand"] == estimand) & (path_plot["cross_fit"] == cross_fit)]
    for loss, g in dd.groupby("loss"):
        gg = g.sort_values("lambda_value")
        ax.plot(gg["lambda_value"], gg["rmse"] ** 2, marker="o", label=loss)
    ax.set_xscale("log")
    ax.set_title(f"{estimand}, cross_fit={cross_fit}")
    ax.set_xlabel("Riesz regularization lambda")
    ax.set_ylabel("MSE")
    ax.legend(loc="best")
fig.suptitle(FIGURE_TITLE_SECTION_3)
fig.tight_layout()
plt.show()